In [1]:
import newkernelo as newker
import kernelo as oldker
import numpy as np
import time
import logging
logging.getLogger().setLevel(logging.INFO)

## Global parameters

In [2]:
gamma_type = 'Full'
sigma_type = 'Diag'
seed = 12345678

# initialisation parameters
gllim_em_iteration = 50
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 1
gmm_em_iteration = 1
gmm_floor = 1e-12
nb_experiences = 10

# training parameters
train_max_iteration = 300
train_ratio_ll = 1e-5
train_floor = 1e-12

##### Generate random dataset

In [3]:
L, D, K, N = 4, 9, 5, 1000

x_gen = np.random.rand(N,L)
y_gen = np.random.rand(N,D)

#### Generate sobol Test Model dataset

In [4]:
L, D, K, N = 4, 9, 5, 100

# Create "physical" model
# dir_path = os.path.dirname(os.path.realpath(__file__))
# print(dir_path+"/pytest/models")
# physical_model = oldker.ExternalModelConfig("RPVModel", "rpv_model", dir_path + "/pytest/models").create()
physical_model = oldker.TestModelConfig().create()

# Create StatModel
covariances = np.random.uniform(0, 0.0001, physical_model.get_D_dimension())
stat_model = oldker.GaussianStatModelConfig("sobol", physical_model, covariances, 12345).create()

# Initialize and train GLLIM model
print("Generating dataset")
x_gen, y_gen = stat_model.gen_data(N)
print(x_gen.shape)
print(y_gen.shape)

Generating dataset
(100, 4)
(100, 9)


#### Generate sobol RPV model dataset

In [5]:
L, D, K, N = 3, 71, 10, 100

# Create "physical" model
# dir_path = os.path.dirname(os.path.realpath(__file__))
# print(dir_path+"/pytest/models")
physical_model = oldker.ExternalModelConfig("RPVModel", "rpv_model", "../../pytest/models").create()

# Create StatModel
covariances = np.random.uniform(0, 0.0001, physical_model.get_D_dimension())
stat_model = oldker.GaussianStatModelConfig("sobol", physical_model, covariances, 12345).create()

# Initialize and train GLLIM model
print("Generating dataset")
x_gen, y_gen = stat_model.gen_data(N)
print(x_gen.shape)
print(y_gen.shape)

Generating dataset
(100, 3)
(100, 71)


## OLD GLLiM

In [6]:
# Create GLLIM model, including its initialization and training configuration
learningConfig = oldker.EMLearningConfig(train_max_iteration, train_ratio_ll, train_floor)
# learningConfig=oldker.GMMLearningConfig(train_max_iteration, train_ratio_ll, train_floor)
# initConfig = oldker.MultInitConfig(seed=123456789, nb_iter_EM=10, nb_experiences=10, gmmLearningConfig=oldker.GMMLearningConfig(15, 10, 1e-12))
initConfig = oldker.MultInitConfig(seed=seed, nb_iter_EM=gllim_em_iteration, nb_experiences=nb_experiences, gmmLearningConfig=oldker.GMMLearningConfig(gmm_kmeans_iteration, gmm_em_iteration, gmm_floor))
gllim_old= oldker.GLLiM(D, L, K, gamma_type, sigma_type, initConfig, learningConfig)


In [7]:

print("Initializing GLLIM model")
gllim_old.initialize(x_gen, y_gen)
gllim_old_params_0 = gllim_old.exportModel() # you can export your gllim model parameters

print("Training model")
gllim_old.train(x_gen, y_gen)
gllim_old_params = gllim_old.exportModel() # you can export your gllim model parameters


INFO:root:Start Multi initialization
INFO:root:Initialisation : 1
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model


Initializing GLLIM model


INFO:root:	Current log likelihood : 108.206070, Best log likelihood : 108.206070
INFO:root:Initialisation : 2
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 102.301820, Best log likelihood : 108.206070
INFO:root:Initialisation : 3
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 138.850166, Best log likelihood : 138.850166
INFO:root:Initialisation : 4
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 107.923522, Best lo

Training model


INFO:root:Iteration : 34, log likelihood : 125.094794
INFO:root:Iteration : 35, log likelihood : 125.094794
INFO:root:Iteration : 36, log likelihood : 125.094794
INFO:root:Iteration : 37, log likelihood : 125.094794
INFO:root:Iteration : 38, log likelihood : 125.094794
INFO:root:Iteration : 39, log likelihood : 125.094794
INFO:root:Iteration : 40, log likelihood : 125.094794
INFO:root:Iteration : 41, log likelihood : 125.094794
INFO:root:Iteration : 42, log likelihood : 125.094794
INFO:root:Iteration : 43, log likelihood : 125.094794
INFO:root:Iteration : 44, log likelihood : 125.094794
INFO:root:Iteration : 45, log likelihood : 125.094794
INFO:root:Iteration : 46, log likelihood : 125.094794
INFO:root:Iteration : 47, log likelihood : 125.094794
INFO:root:Iteration : 48, log likelihood : 125.094794
INFO:root:Iteration : 49, log likelihood : 125.094794
INFO:root:Iteration : 50, log likelihood : 125.094794
INFO:root:Iteration : 51, log likelihood : 125.094794
INFO:root:Iteration : 52, lo

### Get theta_0 from oldker

In [8]:
# theta_0 = newker.GLLiMParameters(L,D,K, "full", "diag")

# print(np.array(gllim_parameters_0.Pi).shape)
# print(np.array(gllim_parameters_0.A).shape)
# print(np.array(gllim_parameters_0.C).shape)
# print(np.array(gllim_parameters_0.Gamma).shape)
# print(np.array(gllim_parameters_0.B).shape)
# print(np.array(gllim_parameters_0.Sigma).shape)

# print(np.array(gllim_parameters_0.Pi))
# print(np.transpose(np.array(gllim_parameters_0.A), (1,2,0)).shape)
# print(np.array(gllim_parameters_0.C).shape)
# print(np.transpose(np.array(gllim_parameters_0.Gamma), (1,2,0)).shape)
# print(np.array(gllim_parameters_0.B).shape)
# print(np.transpose(np.array(gllim_parameters_0.Sigma), (1,2,0)).shape)

# theta_0.Pi = np.copy(np.array(gllim_parameters_0.Pi))
# theta_0.A = np.copy(np.transpose(np.array(gllim_parameters_0.A), (1,2,0)))
# theta_0.C = np.copy(np.array(gllim_parameters_0.C))
# # theta_0.Gamma = np.copy(np.transpose(np.array(gllim_parameters_0.Gamma), (1,2,0)))
# theta_0.Gamma = np.copy(gllim_parameters_0.Gamma)
# theta_0.B = np.copy(np.array(gllim_parameters_0.B))
# # theta_0.Sigma = np.copy(np.transpose(np.array(gllim_parameters_0.Sigma), (1,2,0)))
# Sigma_diag = np.zeros((K,D))
# for k in range(K):
#     Sigma_diag[k,:] = np.diag(gllim_parameters_0.Sigma[k])
# theta_0.Sigma = Sigma_diag

# print(np.sum(theta_0.Pi))
# gllim.setParams(theta_0)

## NEW GLLiM

In [9]:
new_gllim = newker.GLLiM(L,D,K,gamma_type.lower(), sigma_type.lower())

verbose = 0
X_gen = x_gen.T
Y_gen = y_gen.T

new_gllim.initialize(X_gen, Y_gen, gllim_em_iteration, gllim_em_floor, gmm_kmeans_iteration, gmm_em_iteration, gmm_floor, nb_experiences, seed, verbose)
new_gllim_params_0 = new_gllim.getParams()

new_gllim.train(X_gen, Y_gen, train_max_iteration, train_ratio_ll, train_floor, verbose)
new_gllim_params = new_gllim.getParams()


INFO: GLLiM Parameters initialized
INFO: GLLiM dimensions are (L=3, D=71, K=10)
INFO: GLLiM constraints are :
	gamma_type = 'full',
	sigma_type = 'diag'.


In [10]:
print(gllim_old_params_0.Pi)
print(new_gllim_params_0.Pi)

print("\n")
print(gllim_old_params.Pi)
print(new_gllim_params.Pi)

print("\n")
print(gllim_old_params.B)
print(new_gllim_params.B)

[0.07 0.12 0.17 0.04 0.08 0.1  0.11 0.21 0.07 0.03]
[[0.11       0.12       0.07999999 0.08       0.09       0.1
  0.1        0.20000011 0.0799999  0.04      ]]


[0.07 0.12 0.17 0.04 0.08 0.1  0.11 0.21 0.07 0.03]
[[0.11       0.12       0.07999999 0.08       0.09       0.1
  0.1        0.20000011 0.0799999  0.04      ]]


[[ 5.53488550e-01  5.61314496e-01  6.32376030e-01  7.60394905e-01
   9.51424237e-01  1.20073804e+00  1.46498805e+00  1.46497915e+00
   1.20079582e+00  9.51312770e-01]
 [ 7.60456888e-01  6.32240971e-01  5.61400383e-01  5.53234910e-01
   1.46119876e-01  2.18363907e-01  2.85271380e-01  3.66736406e-01
   4.79019731e-01  6.41752340e-01]
 [ 8.77626836e-01  1.20083524e+00  1.58785316e+00  1.87370543e+00
   1.67112772e+00  1.46497887e+00  1.33915294e+00  1.36971943e+00
   3.82754172e+00  3.57479203e+00]
 [ 3.54326146e+00  2.43280305e+00  1.67115983e+00  1.12415644e+00
   7.60415808e-01  5.24296287e-01  3.67034601e-01  2.53469960e-01
   1.60031143e-01  6.46035121e-02]
 [-6.5

In [11]:
# Y_gen = np.array(y_gen[:5].T)
# pred = new_gllim.inverseDensities(Y_gen)
# print(pred.meanPredResult.mean)
# print(x_gen[:5])
# print("\n")


predicator = oldker.PredictionConfig(2, 2, 1e-10, gllim_old).create()
for i in range(5):
    print(x_gen[i])

    prediction = predicator.predict(y_gen[i], np.zeros(D))
    print(prediction.meansPred.mean)

    pred = new_gllim.inverseDensities(y_gen[i])
    print(pred.meanPredResult.mean)
    print("\n")


[0.5 0.5 0.5]
[0.6256874  0.57058902 0.49221973]
[[0.46441409 0.2952956  0.45539875]]


[0.75 0.25 0.25]
[0.75264704 0.20445443 0.2343928 ]
[[0.75529593 0.21616291 0.23706719]]


[0.25 0.75 0.75]
[0.84954205 0.75385914 0.89968915]
[[0.19431336 0.77123179 0.62277403]]


[0.375 0.375 0.625]
[0.36472681 0.35871869 0.62201137]
[[0.50373767 0.18911474 0.71163173]]


[0.875 0.875 0.125]
[0.56761018 0.95324026 0.14178784]
[[0.56761018 0.95324026 0.14178784]]




In [12]:
pred.centerPredResult.means

array([], shape=(0, 0), dtype=float64)

In [13]:
insights = new_gllim.getInsights()

In [14]:
print(insights.time)

0:00:03
